# Notebook 1A: Introduction to Machine Learning and Exploring the METABRIC Dataset

*Authored by Dr. Noelle Anderson in 2026*

## Introduction

In this notebook, you will get oriented to the machine learning landscape and start working with a real breast cancer patient dataset called **METABRIC**. You will load the dataset, inspect its structure, identify different data types, explore missing data, and carry out some basic preprocessing.

Why does this matter? In biology, biochemistry, and public health, datasets are often messy mixtures of clinical measurements, categorical variables, and molecular data. Before we can really get into machine learning, we need to understand what kind of data we have and do some basic preparation.

By the end of this notebook, you should be able to:
- describe the difference between supervised learning, unsupervised learning, and generative AI
- explain what tabular data is
- load and inspect the METABRIC dataset
- identify data types and examine missing values
- apply a few basic preprocessing steps
- save a cleaned version of the dataset for the next notebook

### The Machine Learning Landscape

Machine learning (ML) can sound very mysterious, but you've likely encountered ML in everyday life: when Netflix recommends a show, when your email filters spam, or when your bank flags a suspicious transaction. But "machine learning" is a broad term, it's important to first understand what we're actually talking about and the terms that often get thrown around.

Artificial intelligence (AI) is a broad term for computer systems that can perform tasks that usually require human-like intelligence, such as recognizing speech, making decisions, understanding language, or generating content, while machine learning is a specific approach *within* AI where the system learns patterns from data instead of being programmed with every rule explicitly.

**Here are three fundamental AI approaches to distinguish:**

- **Classical Machine Learning**: You build models to find patterns in structured data (like spreadsheets) and make predictions. This is what we'll focus on in this course!
  - Example: Predicting whether a patient has a disease based on their lab measurements, or forecasting house prices based on size, location, and amenities.

- **Deep Learning**: A subset of machine learning that uses more complex models called neural networks with many layers to find complex patterns, especially in unstructured data like images, audio, or text.
  - Example: Recognizing faces in photos, transcribing speech to text, or translating languages.

- **Generative AI**: Models that create new text, images, audio, or other media—based on patterns learned from existing (and often stolen) data. This includes tools like ChatGPT, Claude, and Gemini.
  - Example: A chatbot composing an email or a model creating an image from a text prompt.


---
**Within classical machine learning, there are two main categories:**

- **Unsupervised learning**: You have data, but no answer column that tells you what to predict. Instead, the goal is to look for patterns, groups, or structure in the data.
  - Example: Grouping tumor samples that look similar based on clinical measurements and gene expression values.
  - We will spend weeks 1 to 3 exploring data with unsupervised learning.

- **Supervised learning**: You have data AND known answer you want a model to learn to predict.
  - Example: Predicting whether a patient has a disease based on their lab measurements and some data to learn from which labels disease or no disease.
  - In weeks 4 to 6, we will focus on using supervised learning for prediction.

This course focuses on **classical machine learning for tabular data**—working with structured datasets organized in rows and columns, like spreadsheets or database tables. This foundation underlies many real-world applications in healthcare, finance, marketing, and research!

Before we get into modeling though, we need to understand and prepare our data!

## Notebook 1A roadmap

In this notebook, we will follow this workflow:

1. **Understand the machine learning landscape**
2. **Load the METABRIC data**
3. **Explore the dataset structure**
4. **Identify data types**
5. **Explore distributions with basic EDA**
6. **Check for and handle missing values**
7. **Encode binary categorical variables**
8. **Save the processed data for Notebook 1B**

## Step 0: Import needed packages

In [ ]:
# These you'll end up loading in all notebooks

import pandas as pd          # for data manipulation and tabular analysis
import numpy as np           # for numerical computing and array operations
import matplotlib.pyplot as plt # for building basic plots and visualizations
import seaborn as sns        # for statistical data visualization and styling

## About the METABRIC dataset

**METABRIC** stands for **Molecular Taxonomy of Breast Cancer International Consortium**. It is a breast cancer research dataset that combines:
- **clinical information**, such as age, tumor size, and receptor status
- **molecular information**, such as gene expression measurements

We are using METABRIC because it is a realistic biomedical dataset. It has a mix of variable types, some missing values, and biologically meaningful measurements. That makes it a good dataset for learning how to inspect and prepare data before we use machine learning methods.

In the next few weeks, we will use this dataset for **unsupervised learning**, which means we will look for patterns and groups in the data without trying to predict a labeled outcome.

**Important:** `patient_id` is the **index** for the dataset. It identifies each patient or sample, but it is **not** a variable that we will use for analysis.

## METABRIC Column Descriptions:

- **age_at_diagnosis**: The age of the patient (in years) when they were diagnosed with breast cancer.

- **neoplasm_histologic_grade**: A measure of how abnormal the tumor cells look under a microscope (1 = more like normal cells, slower growing; 3 = very abnormal, more aggressive).

- **inferred_menopausal_state**: Whether the patient is pre-menopausal or post-menopausal, based on clinical information.

- **lymph_nodes_examined_positive**: The number of lymph nodes found to contain cancer cells, indicating how much the cancer has spread.

- **er_status**: Indicates whether the tumor has estrogen receptors (Positive = tumor may grow in response to estrogen; Negative = does not rely on estrogen).

- **her2_status**: Indicates whether the tumor overexpresses the HER2 protein (Positive = faster-growing, more aggressive tumors; Negative = normal levels).

- **pr_status**: Indicates whether the tumor has progesterone receptors (Positive = tumor may grow in response to progesterone; Negative = does not rely on progesterone).

- **tumor_size**: The size of the primary tumor, usually measured in millimeters.

- **tumor_stage**: A summary of how advanced the cancer is (higher stages indicate larger tumors and/or more spread).

- **15 gene expression columns**: Genes related to breast cancer have been pre-selected, and the column names are the gene names. The values are **gene expression z-scores**, which tell us whether a gene's activity in a tumor sample is **higher than average**, **lower than average**, or **about average** compared with the rest of the dataset.

## Step 1: Load the METABRIC data

Tabular data like METABRIC usually have:
- **rows**, which represent samples or patients
- **columns**, which represent variables or, in machine learning lingo, **"features"**.

Our first job is to load the dataset and confirm that it looks like we expect and get to know it a little bit.

### Predict before running

Before running the next cell, answer these questions in your head:

- Based on the descriptions of the columns above, what datatypes do you expect them to be?

In [ ]:
# Public Google Drive file ID provided by staff
file_id = "1S80owtARAzxX2yrbPUI4xkOG_ID3npTW"

# Direct download link for pandas using an f-string to plug in the file ID
url = f"https://drive.google.com/uc?export=download&id={file_id}"

# Load the dataset with pandas' read_csv
# setting index_col to the first column, patient_id, a unique id per patient we need to keep track of
df = pd.read_csv(url, index_col="patient_id")

*Tip: It's a good idea to pay attention to how we input and output data. There are many different ways to do it, the method above is useful when the data is available by a public link.*

## Step 2: Explore the dataset structure

Before doing machine learning, we usually do some **exploratory data analysis (EDA)**. EDA means taking a first look at the dataset so we can understand what kind of information it contains and what preparation it may need. We cannot work with data we haven't put our own eyes on and understand.

EDA can include things like:
- checking the size and structure of the dataset
- looking at data types
- identifying feature types
- exploring distributions
- checking for missing values

We will do those pieces step by step. In this section, we will start by looking at the basic structure of the dataset.

In [ ]:
# Quick checks: It's important to get to know your data first!
print("Shape:", df.shape)
print("Index name:", df.index.name)

Confirm that your dataset has over 1900 rows and 24 columns, and that `patient_id` is the index.

### <font color=0D9488>**Question 1:**</font> Print the first 5 rows of the dataset and the list of the first 9 column names (the clinical data columns).

*The `.head()` function is useful here!*

In [ ]:
# Your solution here!

Notice here that each row is a patient, and each column contains attributes about that patient.

Notice the types of columns visually, and below explore them programatically:

## Step 3: Identify feature types

Now that we have looked at the basic structure of the dataset, the next step is to identify what kinds of variables we are working with. This matters because different types of variables are explored and prepared in different ways.

Some common feature types in this dataset are:

- **continuous numeric**: values can vary across a range, such as age or tumor size
- **binary categorical**: values come from two categories, such as Positive or Negative
- **ordinal**: values have a meaningful order, such as histologic grade 1, 2, 3
- **gene expression values**: numeric values that tell us whether a gene is expressed above or below average

*Note: Continuous variables are measured on a numeric scale and can take many possible values, while categorical variables place observations into groups or labels such as yes/no or low/medium/high.*


In [ ]:
# Inspect column names and data types with .info()
df.info()

`.info()` is a very useful exploratory function, it shows us not only column names but the count of non-missing ("null") values and the type of data encoded in each column.

### <font color=0D9488>**Question 2:**</font> Use the output above to identify one example of each of the following: a continuous numeric feature, a binary categorical feature, and an ordinal feature.

**Your solution here!**

We see most of our columns are float64 type, meaning they store numeric values with decimals. But several are labeled object; this usually means those columns contain text or mixed kinds of values. Let's take a closer look at the values of those columns:

In [ ]:
# Find the columns stored as object type using select_dtypes function
object_columns = df.select_dtypes(include="object").columns

# Look at the unique values in each object column using a short for loop
for col in object_columns:
    print(f"\n{col}") # prints column name
    print(df[col].dropna().unique()) # prints unique values

We also have a couple columns that are encoded numerically but are actually ordinal categorical data; we can explore those unique values below by specifying them by name:

In [ ]:
catcols = ["neoplasm_histologic_grade", "tumor_stage"]

### <font color=0D9488>**Question 3:**</font> Write a for loop to print the unique values (categories) for the columns in `catcols` above.

In [ ]:
# Your solution here!

## Step 4: Explore data with basic EDA

Now that we know the structure of the dataset and the types of variables it contains, we can start looking at what the values actually look like. This is another part of **exploratory data analysis (EDA)**.

In this section, we will:
- summarize numeric columns
- make a few simple plots
- look for common values, spread, and unusual patterns

First, the `.describe()` function is a quick way to look at how numeric data is distributed:

In [ ]:
# Summary statistics for numeric columns
display(df.describe())

But plotting can make this even clearer. We will practice together below!

Let's first plot a **histogram** using the seaborn package to view the **distribution** of ages in our data. Histograms show how values in a numeric variable are spread out by grouping them into ranges and counting how many observations (in our case, patients) fall in each range.

In [ ]:
# Histogram of age at diagnosis
sns.histplot(data=df, x="age_at_diagnosis", bins=20)
# bins controls how broken up the data is on the x-axis, try different values

plt.title("Histogram Distribution of Age at Diagnosis")
plt.xlabel("Age at diagnosis (years)")
plt.ylabel("Count")
plt.show()

### <font color=0D9488>**Question 4:**</font> Change the number of bins higher and lower and replot each time. What do you notice when there is a smaller number of bins? What do you notice when the bin number is very large?

**Your solution here!**

We can view the same data with a different type of plot, a **boxplot**, or sometimes called a box-and-whiskers plot.

In a boxplot, the line inside the box around y = 60 shows the median, the box shows the middle 50% of the data, and the whiskers show how far the remaining non-outlier values spread. Points far away from the others may be shown separately and can suggest outliers, which are values that are unusually high or low compared with most of the data.

Histograms are useful when you want to see the overall shape of the data, while boxplots are helpful for a quick summary of the center, spread, and unusual (possible outlier) values.

In [ ]:
sns.boxplot(data=df, y="age_at_diagnosis")
plt.title("Boxplot Distribution of Age at Diagnosis")
plt.ylabel("Age at Diagnosis")
plt.show()

### <font color=0D9488>**Question 5:**</font> Now plot `tumor_size` using either a histogram or a boxplot, your choice. Make sure to label your plot appropriately. Notice the shape of the distribution (this will matter to us later!)

In [ ]:
# Your solution here!

Besides histograms and boxplots, we can also use simple **barplots** when we want to count how many observations fall into each category instead of looking at the spread of a numeric variable.

This is useful for any categorical variable, whether it has two categories or many, because it helps us see which groups are most common and whether some categories are much more frequent than others.

Below we'll look at the estrogen receptor status column to see how many of our patients have tumors that are negative or positive for estrogen receptors (and thus how many of them have tumors that may grow in response to estrogen or not).


In [ ]:
# Here we use seaborn's "countplot" function
sns.countplot(data=df, x="er_status")

plt.title("Counts of ER Status")
plt.xlabel("ER status")
plt.ylabel("Count")
plt.show()

### <font color=0D9488>**Question 6:**</font> Use a barplot to print the number of patients who are inferred to be pre vs post menopausal. Make sure to label your plot appropriately (it's fine to leave the x-axis as 0 and 1 for exploration).

In [ ]:
# Your solution here!

Finally, let's take a peek at our gene expression columns to better understand that data. We will use a barplot because here what is interesting is how genes are expressed compared to each other, so it's nice to have them side by side.

In [ ]:
# Save gene expression columsn to a list by their column indices
gene_expression_cols = df.columns[8:].tolist()

# Make a boxplot for several gene expression columns at once by giving it the list
sns.boxplot(data=df[gene_expression_cols])

plt.title("Boxplots of Tumor Gene Expression Z-scores")
plt.xlabel("Gene")
plt.ylabel("Gene expression z-score")
plt.xticks(rotation=45)
plt.show()

Notice how all our genes are centered at 0? That is because these values have been put on a z-score scale, common for expression data. When we use z-score scaling, it means each gene's expression has been standardized so that 0 means about average expression for that gene, positive values mean higher-than-average expression, and negative values mean lower-than-average expression.

This makes it easier to compare patterns across different genes, even if their original measurement scales were different. The points far above or below the whiskers are possible outliers, meaning samples where a gene is expressed much more or much less than usual compared with the rest of the dataset.

## Step 5: Check for and handle missing values

Real data often have missing values. Some measurements may not have been collected, recorded, or available for every patient.

Missing data matters because many machine learning methods cannot handle missing values directly. Before we can use this dataset in later notebooks, we need to identify missing values and make a simple plan for handling them.

In [ ]:
# Count missing values in each column
missing_counts = df.isna().sum().sort_values(ascending=False)

display(missing_counts)

# Also show percentages
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_percent.round(2))

Above, you can see that only a few columns have missing values.

A simple ML rule of thumb is:

- if a column has only a small amount missing, about under 5%, we often fill in those values and keep the column
- if a column has a larger amount missing, like over 20% to 30%, we often drop that column unless it contains information we cannot afford to lose
- for features in between, the decision depends on why the data is missing and what the analysis needs

Filling in missing values is called **imputation**. Simple methods include filling missing values with the mean, median, or mode of a column. These methods are easy to use and often work well enough as a first step, but they are not perfect because they can reduce real variation in the data.

For now, we will use simple choices only so the cleaned dataset is ready for later notebooks. You will learn more formal imputation methods later.

### <font color=0D9488>**Question 6:**</font> Which 1 column should we drop based on the percent missing? Drop the column in your df and keep the df variable name as is.

In [ ]:
# Your solution here!

Now we do not have to handle the missing values, but there are still two columns with missing values.

For `neoplasm_histologic_grade`, mode imputation makes sense as a simple beginner choice because grade is an ordered category, not a continuous measurement. The mode is the most common observed value.

For `tumor_size`, median imputation makes sense as a simple beginner choice because tumor size is continuous and may be skewed by a few large values.

These choices are not the only possible choices. We are using them here to create a complete dataset for the next notebooks.

In [ ]:
# Simple imputation demostrated here

# Mode imputation for an ordered categorical feature
mode_grade = df["neoplasm_histologic_grade"].mode()[0]
df["neoplasm_histologic_grade"] = df["neoplasm_histologic_grade"].fillna(mode_grade)

# Median imputation for a continuous feature
median_size = df["tumor_size"].median()
df["tumor_size"] = df["tumor_size"].fillna(median_size)

print(df.isna().sum().sum())

If you see a 0 above, you're ready to move on!

## Step 6: Encode binary categorical features

Now we need to deal with a different data prep issue: some columns contain **categories written as words instead of numbers**. In this dataset, those are `er_status`, `pr_status`, `her2_status`, and `inferred_menopausal_state`.

That is easy for humans to read, but many data analysis and machine learning methods work better when values are stored numerically. Because of that, we often **encode** binary categorical columns by replacing the text labels with numbers.

A simple approach for Positive/Negative columns is:
- `Positive -> 1`
- `Negative -> 0`

And for the `inferred_menopausal_state` column:
- `Pre -> 0`
- `Post -> 1`

This changes the **representation** of the data so models can work with them, but not the underlying biological meaning. We can use simple Python dictionaries to map the encoding, then apply it with the `.map()` command:

In [ ]:
# Make mapping dictionaries
status_map = {"Negative": 0, "Positive": 1}
menopause_map = {"Pre": 0, "Post": 1}

In [ ]:
# Now map the status_map to the Positive/Negative columns
for col in ["er_status", "pr_status", "her2_status"]:
    if col in df.columns:
        df[col] = df[col].map(status_map)

# print the first 5 rows of each column of interest so we can see the difference
# they should be binary, only 0s and 1s
df[["er_status", "pr_status", "her2_status"]].head()

### <font color=0D9488>**Question 7:**</font> Encode the other binary categorical column, `inferred_menopausal_state`. Then print the unique values for those columns again to confirm the encoding worked.

*Hint: remember the pandas command `.unique()`*

In [ ]:
# Your solution here!

Let's do a final check before we save our preprocessed data for analysis in the next notebook.

### <font color=0D9488>**Question 8:**</font> Use the command that prints both non-null count and datatype to check that there is no missing data and that all datatypes are numeric (float or int).

In [ ]:
# Your solution here!

## Step 7: Save the processed data

It is often useful to save an intermediate version of the dataset after basic preprocessing. That way, later notebooks can start from a clean, consistent file instead of repeating the same steps.

To save a dataframe as a CSV file in our google drive, we first need to mount your drive:

In [ ]:
# Must import drive to access your Google Drive
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Then we can tell it where we want it to save your file, a "path" or an "address" for your file. If you don't already have a folder called `SCIP2026`, the makedirs function from the os package below will make one.


In [ ]:
# Name the folder where we want to save our course files, let's all call it SCIP2026ML for consistency
path = "/content/drive/MyDrive/SCIP2026ML"

# Must import os to see if the file folder already exists or if we need to make one
import os

# Create the folder if it does not already exist
os.makedirs(f"{path}", exist_ok=True)

Then we can actually name and save our file with to_csv, in the path location we specified above.

In [ ]:
# Save the processed dataset to our folder

# Name the file we want to save
dfname = "metabric_1a_cleaned.csv"

# Save the processed dataset into that folder
df.to_csv(f"{path}/{dfname}")

print(f"Saved your cleaned file to: {path}/{dfname}")

*Tip: It's a good idea to pay attention early to how files are input and output!*

### <font color=0D9488>**Question 9:**</font> Below, write a list of the new things you learned today, both concepts and key coding functions. Add brief definitions/descriptions and notes to help you remember the key points.

*We also recommend you keep these in a separate document to make a running cheatsheet for the entire course; this is often very helpful for students!*

# Congratulations, you have completed the first notebook!

In this notebook, you took your first steps in the machine learning workflow by working with a real breast cancer dataset, **METABRIC**!

You learned that machine learning includes different kinds of tasks, including **supervised learning**, **unsupervised learning**, and **generative AI**, and that this course will focus on finding patterns in structured biomedical data.

You loaded the METABRIC dataset, inspected its structure, identified different feature types, used plots to explore how values are distributed, checked for missing data, and applied a few basic preprocessing steps to get the data ready for later analysis.

You also saw that preparing a dataset requires careful choices about what each column means, how missing values should be handled, and how categorical information should be represented numerically.

## Where this fits in the machine learning workflow

This notebook focused on the **early stages** of a machine learning workflow. Before we can build models or look for patterns, we need to understand the dataset and make sure it is in a usable form. In this notebook, that meant:

- **loading in the data**
- **exploring its structure and feature types**
- **visualizing important variables with histograms, barplots, and boxplots**
- **checking and handling missing values**
- **encoding binary categorical features**
- **saving a cleaned version of the dataset**

These steps are important because machine learning methods work best when the data has already been carefully inspected and prepared. In the next notebook, we will build directly on this cleaned dataset and use a method called **PCA** to start looking for structure and patterns in the METABRIC data, see you there!